In [5]:
import json
from transformers import AutoTokenizer
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig

def compile_predictions(predictions_file, model_name):
    extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    results = []
    total_tokens = 0
    with open(predictions_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            gold = parse(f"${data['answer']}$", extraction_config=extraction_target)
            
            # Get model generation output (usually first item)
            llm_output = data['model_generation'][0] if isinstance(data['model_generation'], list) else data['model_generation']
            answer = parse(llm_output, extraction_config=extraction_target)
            total_tokens += len(tokenizer.encode(llm_output))
            result = verify(gold, answer)
            results.append(result)
    
    accuracy = sum(results) / len(results) if results else 0
    avg_tokens = total_tokens / len(results) if results else 0
    return accuracy, avg_tokens

def compile_predictions_passK(predictions_file, model_name):
    extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    results = []
    total_tokens = 0
    with open(predictions_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            gold = parse(f"${data['answer']}$", extraction_config=extraction_target)
            
            o_results = []
            # Use all generations to verify
            llm_total_tokens = 0
            for llm_output in data['model_generation']:
                answer = parse(llm_output, extraction_config=extraction_target)
                result = verify(gold, answer)
                o_results.append(result)
                llm_total_tokens += len(tokenizer.encode(llm_output))
            total_tokens += llm_total_tokens/len(data['model_generation'])
            results.append(o_results)

    # pass@K means at least one correct
    passK = sum(1 for o_result in results if any(o_result)) / len(results) if results else 0
    avg_tokens = total_tokens / len(results) if results else 0
    
    return passK, avg_tokens


In [2]:
import matplotlib.pyplot as plt

def draw_comparison_fig(baseline_x, baseline_y, seal_x, seal_y, x_label, y_label, title):
    plt.figure(figsize=(10, 5))
    plt.plot(baseline_x, baseline_y, marker='o', linestyle='-', color='b', label='Baseline')
    plt.plot(seal_x, seal_y, marker='o', linestyle='-', color='r', label='SEAL')
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.ylim(bottom=0, top=1)
    plt.legend()
    plt.show()

## Example

In [7]:
import os
paths = ["/media/volume/llm/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_test/16/0.7/8192/predictions.jsonl",
         "/media/volume/llm/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/MATH500/16/0.7/8192/predictions.jsonl"]
for path in paths:
    metric_path = os.path.join(os.path.dirname(path), "metrics.json")
    accuracy, avg_tokens = compile_predictions(path, "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
    passK, avg_tokens = compile_predictions_passK(path, "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
    print("accuracy: ", accuracy, "passK: ", passK, "avg_tokens: ", avg_tokens)
    # write to metric_path
    with open(metric_path, 'w') as f:
        json.dump({"accuracy": accuracy, "passK": passK, "avg_tokens": avg_tokens}, f)


accuracy:  0.8248673237300985 passK:  0.9658832448824868 avg_tokens:  1462.8264310083396
accuracy:  0.794 passK:  0.95 avg_tokens:  3784.126625
